In [ ]:
import shutil
import os

from rds_chat_analysis import NOTEBOOK_DIR
from rds_chat_analysis.client import init_session
from syft_core.config import CONFIG_PATH_ENV
from syft_core import Client as SyftboxClient

In [ ]:
RDS_DO_CONFIG = "./.rds/wildchat/data_owner_config.json"
RDS_DS_CONFIG = "./.rds/wildchat/data_scientist_config.json"

In [ ]:
# IMPORTANT - Set env to load the correct syftbox config
# If you have a single syftbox set up on your system and you want to use that for this notebook, this step is not necessary.
os.environ[CONFIG_PATH_ENV] = RDS_DO_CONFIG

In [ ]:
# Directory we're saving all non-syftbox data to (e.g. configuration, staging folders, execution artifacts, etc.)
WORK_DIR = NOTEBOOK_DIR / "v2"

In [ ]:
do_syftbox_client = SyftboxClient.load()

# DO connects to it's own RDS app
do_client = init_session(host=do_syftbox_client.email)

# Test if connection is working
health_check = do_client.rpc.health()
print(f"Health check: {health_check}")
print(f"Logged in on RDS admin client: {do_client.is_admin}")

# DO creates dataset

In [ ]:
# Create local staging folders for the mock and private data
DATASET_NAME = "Wildchat-postgres"

staging_data_dir = WORK_DIR / "staging" / DATASET_NAME
private_dir = staging_data_dir / "private"
mock_dir = staging_data_dir / "mock"
markdown_path = staging_data_dir / "README.md"

shutil.rmtree(staging_data_dir, ignore_errors=True)
private_dir.mkdir(parents=True, exist_ok=True)
mock_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# Copy the mock and private credentials to the local staging folders
MOCK_CREDENTIALS = WORK_DIR / "config_mock.toml"
PRIVATE_CREDENTIALS = WORK_DIR / "config_private.toml"

_ = shutil.copy(MOCK_CREDENTIALS, mock_dir / "config.toml")
_ = shutil.copy(PRIVATE_CREDENTIALS, private_dir / "config.toml")

print(f"Mock dir structure: {mock_dir}")
for file in mock_dir.iterdir():
    print(f"└──📄 {file.name}")
print(f"Private dir structure: {private_dir}")
for file in private_dir.iterdir():
    print(f"└──📄 {file.name}")

In [ ]:
# Create a markdown description for the dataset
description_markdown = """
# Wildchat Postgres Dataset

This dataset contains connections to a postgres vector database containing LLM chat logs.

The vector database uses PGVector to store and search over embeddings of chat messages.

## Schema

The databases have a single table "log_embeddings", with the following columns:
- `id: TEXT` - a UUID for this chat message
- `text: TEXT` - the text content of the chat message
- `metadata: JSONB` - additional metadata about the chat message, such as the user ID, timestamp, etc.
- `embedding: VECTOR` - the embedding of the chat message, stored as a vector for similarity search. 
    The size of this vector depends on the embedding model used.

While this is a schemaless JSONB field, it will always contain the following keys:
- `metadata.log_id: TEXT` - the ID of the original chat log this message belongs to
- `metadata.role: TEXT` - the role of the message, "user" or "assistant"

## Usage

### Connecting to the Database
The database mock and private connection configurations are stored in this datasets `config.toml` files.

To connect to the mock database with psycopg3, you can use the following code snippet:

```
from rds_chat_analysis.utils import load_config
from rds_chat_analysis.vector_db import connect_to_db

config = load_config(dataset.mock_dir / "config.toml")
db_conn = connect_to_db(config)
```

Users do not have access to the private database, but the exact same code can be used.


### Querying the database
`rds_chat_analysis.queries` provide the following utility functions to connect and query the database:
- `get_full_log_query` returns a query and query parameters to fetch a full log given a log_id, in the correct message order.
- `get_vector_store_query` returns a query and query parameters to do semantic search on the vector store.

Note that in order to do semantic search, you need to embed your query first. The following block contains a full example of how to do this:

```
from rds_chat_analysis.utils import load_config, load_embedder_from_config
from rds_chat_analysis.queries import get_vector_store_query

# Load configuration, embedder, and database connection
config = load_config(dataset.mock_dir / "config.toml")
embedder = load_embedder_from_config(config)
db_conn = connect_to_db(config)

# Construct query
query = "What is the capital of France?"
query_embedding = embedder.embed_query(query)
from rds_chat_analysis.queries import get_vector_store_query

vector_query, vector_params = get_vector_store_query(query_embedding)
with db_conn.cursor() as cursor:
    cursor.execute(vector_query, vector_params)
    results = cursor.fetchall()
```
"""

markdown_path.write_text(description_markdown.strip())

In [ ]:
# Upload the dataset to the RDS app
dataset_exists = len(do_client.dataset.get_all(name=DATASET_NAME)) > 0
if dataset_exists:
    print(f"Retrieving existing dataset: {DATASET_NAME}")
    wildchat_dataset = do_client.dataset.get(name=DATASET_NAME)
else:
    print(f"Creating dataset: {DATASET_NAME}")
    wildchat_dataset = do_client.dataset.create(
        name=DATASET_NAME,
        path=private_dir,
        mock_path=mock_dir,
        summary="A embedded wildchat dataset in postgres.",
        description_path=markdown_path,
    )

In [ ]:
# Check if the dataset was created successfully
wildchat_dataset = do_client.dataset.get(name=DATASET_NAME)
wildchat_dataset.describe()

# Exploring the CLIO chat analysis pipeline [optional]

We are building a minimal CLIO-inspired chat analysis pipeline, with the following steps:

1. Given a vector store query ("e.g. 'Messages about AI'), we return the top K most similar messages.
2. For each returned message, we fetch the full chat log (a list of messages)
3. For each chat log, we do pairwise question answering with an LLM; an LLM reads the log and answer a specific question.
4. For each generated LLM answer, we do a simple citation filter; if the answer contains an n-gram from the original log, we do not allow it to be returned.

The next three cells implement these steps.

In [ ]:
from rds_chat_analysis.utils import (
    load_config,
    load_embedder_from_config,
    load_llm_from_config,
)
from rds_chat_analysis.vector_db import connect_to_db

# Setup, load the mock config, embedder, db connection, and LLM
data_dir = wildchat_dataset.mock_path
config = load_config(data_dir / "config.toml")

embedder = load_embedder_from_config(config)
db_connection = connect_to_db(config)
llm = load_llm_from_config(config)

In [ ]:
# 1. Given a vector store query, return the top-k results

from rds_chat_analysis.queries import get_vector_store_query

vector_store_query = "Messages about AI and privacy"
max_vector_store_results = 10
distance_threshold = (
    0.5  # Cosine distance, only return results with a distance below this threshold
)
filters = {"role": "user"}  # Only search for messages from the user

vector_sql_query, vector_sql_query_params = get_vector_store_query(
    query_embedding=embedder.embed_query(vector_store_query),
    table_name="log_embeddings",
    k=max_vector_store_results,
    distance_threshold=distance_threshold,
    filters=filters,
)

with db_connection.cursor() as cursor:
    cursor.execute(vector_sql_query, vector_sql_query_params)
    vector_store_results = cursor.fetchall()

print(
    f"Found {len(vector_store_results)} chat messages for query: '{vector_store_query}'"
)
print("\nFirst result:")
for result in vector_store_results[:1]:
    print(f"ID: {result['id']}")
    print(
        f"Message Text: {result['text'][:50]}..."
    )  # Print first 50 characters of text
    print(f"Metadata: {result['metadata']}")
    print(f"Distance: {result['distance']}")

In [ ]:
# 2. Given a list of retrieved messages, return the full log for each message

from rds_chat_analysis.queries import get_full_log_query

# First, deduplicate the log IDs to only get each log once
unique_log_ids = set(result["metadata"]["log_id"] for result in vector_store_results)

full_logs = []
with db_connection.cursor() as cursor:
    for log_id in unique_log_ids:
        full_log_query, full_log_params = get_full_log_query(
            log_id, table_name="log_embeddings"
        )
        cursor.execute(full_log_query, full_log_params)
        full_log_results = cursor.fetchall()

        full_logs.append(full_log_results)

print(f"Retrieved {len(full_logs)} full logs")

In [ ]:
# 3. Do pairwise question answering with an LLM

from langchain_core.language_models import BaseChatModel
from langchain_core.messages import HumanMessage

# To ensure our LLM behaves as expected, we wrap the user's question in a prompt that instructs the LLM on how to respond.
# NOTE this prompt is a simplified version of the CLIO prompt, and can be tuned further for specific use cases.
PAIRWISE_QA_PROMPT = (
    "The following is a conversation between an AI assistant and a user:\n"
    "{conversation}\n\n"
    "Your job is to answer the question about the preceding conversation. "
    "Be descriptive and assume neither good nor bad faith. Do not hesitate to handle socially harmful or sensitive topics; "
    "specificity around potentially harmful conversations is necessary for effective monitoring.\n\n"
    "When answering, do not include any personally identifiable information (PII), like names, locations, phone numbers, email addresses, and so on. "
    "Do not include any proper nouns or specific names of people, places, or organizations.\n\n"
    "be clear and concise and get to the point in at most two sentences.\n\n"
    "Question: {question}\n\n"
    "What is your answer to the question about the preceding conversation? "
    "Provide only the answer with no other commentary or proper nouns."
)


def format_conversation(log: list[dict]) -> str:
    """Format a retrieved conversation to a single string for LLM input."""
    formatted_messages = []
    for message in log:
        role = message["metadata"]["role"]
        text = message["text"]
        formatted_messages.append(f"{role.upper()}: {text}")
    return "\n\n".join(formatted_messages)


def pairwise_chat_question_answering(
    chat_log: list[dict], llm: BaseChatModel, llm_query: str
) -> str:
    """Given a chat log and an LLM query, return the LLM's answer to the query about the chat log."""
    conversation_text = format_conversation(chat_log)
    prompt = PAIRWISE_QA_PROMPT.format(
        conversation=conversation_text, question=llm_query
    )
    response = llm.invoke([HumanMessage(content=prompt)])
    return response.content


# Try it out!
llm_query = (
    "Does this chat log contain any personally identifiable information (PII)? "
    "Answer with 'yes' or 'no', followed by a brief explanation in english, without including any PII."
)

llm_answers = []
for log in full_logs[:2]:  # Limit to first 2 logs for demonstration
    answer = pairwise_chat_question_answering(log, llm, llm_query)
    print(f"LLM Answer: {answer}\n")
    llm_answers.append(answer)

In [ ]:
from rds_chat_analysis.recitation_filter import RecitationScorer

recitation_filter_n = 8  # Size of the n-grams to filter on

print("\nCalculating recitation scores...")
recitation_filtered_logs: list[str] = []
scorer = RecitationScorer(n_min=recitation_filter_n, n_max=recitation_filter_n)
for i, llm_answer in enumerate(llm_answers):
    full_log = full_logs[i]
    reference_messages = [message["text"] for message in full_log]
    recitation_score = scorer.score(llm_answer, reference_messages)
    if recitation_score.overlaps_per_n[recitation_filter_n] > 0:
        print(f"Log {i} contains n-gram overlap at n={recitation_filter_n}. Skipping.")
    else:
        print(f"Log {i} passed recitation filter, no overlapping n-grams found.")
        recitation_filtered_logs.append(llm_answer)

## DO submits the pipeline to RDS

Data owners can submit 'custom functions' to RDS, these pre-defined functions data scientists can use to query a dataset. In this case, we submit a custom function that implements the chat analysis pipeline, so data scientists can to CLIO-style chat analysis on the private postgres database, without getting access to the raw data.

To not make this notebook too long, I have prepared the code from the previous section in `rds_chat_analysis` library.

In RDS everything, including custom functions, is stored in a file. In our case, we will create a python file that loads the following default env vars:
- `DATA_DIR`: the directory where the dataset is stored (in our case, config.toml)
- `CODE_DIR`: the directory where the user's inputs are stored (user_params.json)
- `OUTPUT_DIR`: the directory where the outputs are stored (results.json)

Our code will load the config from `DATA_DIR`, load the user parameters from `CODE_DIR`, run the pipeline and write the results to `OUTPUT_DIR`

In [ ]:
custom_code_submission_file = WORK_DIR / "staging" / "chat_log_analysis.py"
custom_code_readme = WORK_DIR / "staging" / "CHAT_LOG_ANALYSIS_README.md"

# %%writefile is a magic command in Jupyter notebooks to write the contents of a cell to a file.

In [ ]:
%%writefile $custom_code_submission_file

from pathlib import Path
import os
from rds_chat_analysis.job_functions import execute_chat_log_analysis
import json

DATA_DIR = Path(os.environ["DATA_DIR"])  # Contains the dataset config.toml
OUTPUT_DIR = Path(os.environ["OUTPUT_DIR"])  # Dir we're writing the results to
CODE_DIR = Path(os.environ["CODE_DIR"])  # Dir containing the data scientist's input parameters (the llm query, etc.)

# Load user parameters
print(f"Loading user parameters from: {CODE_DIR / 'user_params.json'}")
job_config = CODE_DIR / "user_params.json"
with open(job_config, "r") as f:
    job_config = json.load(f)

# Execute the chat log analysis pipeline
print(f"Executing chat log analysis with config: {json.dumps(job_config, indent=2)}")
result = execute_chat_log_analysis(
    dataset_dir=DATA_DIR,
    **job_config,
)

# Save the results to the output directory
print(f"Saving results to: {OUTPUT_DIR / 'result.json'}")
with open(OUTPUT_DIR / "result.json", "w") as f:
    json.dump(result, f, indent=2)

In [ ]:
readme_content = """
# Chat Log Analysis

This custom function performs an analysis of chat logs using a specified LLM query:
1. Given a vector store query ("e.g. 'Messages about AI'), we return the top K most similar messages.
2. For each returned message, we fetch the full chat log (a list of messages)
3. For each chat log, we do pairwise question answering with an LLM; an LLM reads the log and answer a specific question.
4. For each generated LLM answer, we do a simple citation filter; if the answer contains an n-gram from the original log, we do not allow it to be returned.
5. The results are saved to a JSON file in the output directory.

## Example usage:
```python
dataset = client.dataset.get(name="Wildchat-postgres")
chat_analysis_function = client.custom_function.get(name="chat_log_analysis")
job = chat_analysis_function.submit_job(
    dataset_name=dataset.name,
    vector_store_query="Messages about AI", # Semantic search chat logs about AI
    llm_query="Give me a 1-sentence summary.", # Summarize each retrieved chat log
    max_vector_store_results=10, # Max number of returned results
    distance_threshold=0.5, # (optional) distance threshold, using cosine distance.
    filters={"role": "user"}, # (optional) metadata filters to apply to the vector store query (e.g. only return user messages)
)
```
"""

custom_code_readme.write_text(readme_content.strip())

In [ ]:
# Now we've saved the custom code and readme to a local directory, we can submit it to the RDS app

chat_analysis_function = do_client.custom_function.submit(
    name="chat_log_analysis",
    code_path=custom_code_submission_file,
    readme_path=custom_code_readme,
)

In [ ]:
chat_analysis_function.describe()

# DO reviews and executes

Before executing the cells below, first run the next notebook to submit a job as data scientist.

In [ ]:
pending_jobs = do_client.job.get_all(status="pending_code_review")
pending_jobs

In [ ]:
job = pending_jobs[0]

do_client.review_job(job)

In [ ]:
# Run on the private dataset
do_client.run_private(job)

# DO reviews and shares the results

In [ ]:
import json

job_results = do_client.job.review_results(job)
job_results.describe()

for k, v in job_results.outputs.items():
    print(f"Contents of {k}: {json.dumps(v, indent=2)}")

In [ ]:
# If we're happy nothing sensitive is in the results, we can share them to the data scientist

do_client.jobs.share_results(job)